# 7장 동등성과 비열등성 검정 (R과 파이썬으로 하는 통계분석)

보건통계학 기초 13장의 예로 평균 차이와 비율 차이의 비열등성 검정, 두 단측 검정과 생물학적 동등성, 교차설계 분석, 비열등성 시험의 표본크기를 R로 계산합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요. 패키지가 없으면 먼저 install.packages("DescTools")를 실행합니다.


## a. 평균 차이의 비열등성

### 셀 1. 원격 상담 시험의 비열등성 검정 (13장 b 절)

In [ ]:
d <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch13_pdc.csv")
remote   <- d$pdc[d$arm == "remote"]          # 원격 상담군 304명
inperson <- d$pdc[d$arm == "inperson"]        # 대면 상담군 304명
t.test(remote, inperson, mu = -5, alternative = "greater", conf.level = 0.975)
t.test(remote, inperson)$conf.int                                    # 양측 95% 구간
t.test(remote, inperson, mu = -5, alternative = "greater")$conf.int  # conf.level을 빼면
t.test(inperson, remote, mu = -5, alternative = "greater")$p.value   # 순서를 바꾸면

### 셀 2. 요약값으로 하는 비열등성 검정 (13장 b·c 절)

In [ ]:
ni_t <- function(m1, s1, n1, m2, s2, n2, margin, higher = TRUE) {   # 요약값으로 하는 Welch 비열등성 검정
  v1 <- s1^2 / n1; v2 <- s2^2 / n2; se <- sqrt(v1 + v2)
  df <- (v1 + v2)^2 / (v1^2 / (n1 - 1) + v2^2 / (n2 - 1))
  d  <- m1 - m2; q <- qt(0.975, df)
  t  <- if (higher) (d + margin) / se else (margin - d) / se     # 한계 쪽 끝을 검정
  round(c(diff = d, lower = d - q * se, upper = d + q * se, t = t, df = df,
          p_NI = pt(t, df, lower.tail = FALSE)), 5)
}
rbind(PDC_ITT  = ni_t(83.9, 16.8, 304, 85.2, 17.5, 304, margin = 5),
      PDC_PP   = ni_t(85.0, 15.9, 262, 85.6, 17.1, 275, margin = 5),
      PDC_ex   = ni_t(85.0, 18.0, 250, 87.5, 17.0, 250, margin = 5),
      HbA1c_FAS = ni_t(-0.74, 1.02, 185, -0.83, 1.06, 183, margin = 0.4, higher = FALSE),
      HbA1c_PP  = ni_t(-0.79, 1.00, 162, -0.85, 1.04, 165, margin = 0.4, higher = FALSE),
      SBP_ex    = ni_t(135.6, 10.6, 150, 132.2, 10.6, 150, margin = 5, higher = FALSE))

## b. 비율 차이의 비열등성

### 셀 3. 제균율 차이의 세 가지 신뢰구간 (13장 d 절)

In [ ]:
suppressPackageStartupMessages(library(DescTools))
BinomDiffCI(186, 224, 190, 224, method = c("wald", "score", "mn"))   # ITT: 신약 186/224, 표준 190/224
BinomDiffCI(180, 205, 185, 208, method = c("wald", "score", "mn"))   # PP

### 셀 4. Farrington–Manning 비열등성 검정

In [ ]:
fm_test <- function(x1, n1, x2, n2, delta) {   # Farrington–Manning 검정, 귀무가설: p1 - p2 = delta
  nll <- function(p2) -(dbinom(x1, n1, p2 + delta, log = TRUE) + dbinom(x2, n2, p2, log = TRUE))
  p2 <- optimize(nll, c(max(0, -delta), min(1, 1 - delta)) + c(1e-9, -1e-9), tol = 1e-12)$minimum
  p1 <- p2 + delta                               # 귀무가설 아래에서 가장 그럴듯한 두 비율
  z  <- (x1 / n1 - x2 / n2 - delta) / sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
  round(c(p1_null = p1, p2_null = p2, z = z, p = pnorm(z, lower.tail = FALSE)), 5)
}
rbind(ITT = fm_test(186, 224, 190, 224, delta = -0.10),
      PP  = fm_test(180, 205, 185, 208, delta = -0.10))

### 셀 5. 성공률이 100%에 가까울 때

In [ ]:
BinomDiffCI(48, 50, 50, 50, method = c("wald", "score", "mn"))     # 소규모 시험 (13장 '결과는 어떻게 읽나?')
BinomDiffCI(94, 100, 98, 100, method = c("wald", "score", "mn"))   # 군당 100명
fm_test(48, 50, 50, 50, delta = -0.10)

## c. 동등성과 생물학적 동등성

### 셀 6. 12명 AUC의 두 단측 검정과 90% 신뢰구간

In [ ]:
R <- c(412, 538, 365, 620, 488, 297, 554, 431, 702, 389, 515, 460)   # 대조약 AUC (13장 e 절의 12명)
T <- c(366, 455, 303, 548, 556, 268, 535, 510, 639, 429, 412, 525)   # 시험약 AUC
lo <- t.test(log(T), log(R), paired = TRUE, mu = log(0.8),  alternative = "greater")
up <- t.test(log(T), log(R), paired = TRUE, mu = log(1.25), alternative = "less")
round(c(t_L = unname(lo$statistic), p_L = lo$p.value, t_U = unname(up$statistic), p_U = up$p.value), 6)
exp(t.test(log(T), log(R), paired = TRUE, conf.level = 0.90)$conf.int)   # 기하평균비의 90% 구간
t.test(log(T), log(R), paired = TRUE)$p.value                           # 차이 = 0의 검정
c(t.test(log(T), log(R), var.equal = TRUE, mu = log(0.8),  alternative = "greater")$p.value,
  t.test(log(T), log(R), var.equal = TRUE, mu = log(1.25), alternative = "less")$p.value)   # 짝을 무시하면

### 셀 7. 2×2 교차설계 분석 (13장 e 절 Table 2)

In [ ]:
be <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch13_be24.csv")              # 대상자 한 명의 한 기간이 한 줄
be$subject     <- factor(be$subject)
be$period      <- factor(be$period)
be$formulation <- factor(be$formulation, levels = c("R", "T"))   # 대조약이 기준
head(be, 4)
fit <- lm(log(auc) ~ subject + period + formulation, data = be)
ci <- confint(fit, "formulationT", level = 0.90)           # 90% 신뢰구간 (로그 척도)
round(100 * exp(c(GMR = unname(coef(fit)["formulationT"]), lower = ci[1], upper = ci[2])), 2)
round(100 * sqrt(exp(sigma(fit)^2) - 1), 1)              # 개체내 변동계수(%)

## d. 표본크기와 검정력

### 셀 8. 평균 차이의 비열등성 표본크기

In [ ]:
z <- qnorm(0.975) + qnorm(0.90)                       # 단측 α 0.025, 검정력 90%
c(HbA1c = 2 * 1.1^2 * z^2 / 0.4^2, PDC = 2 * 18^2 * z^2 / 5^2)   # 정규분포 공식(군당)
power.t.test(delta = 0.4, sd = 1.1, sig.level = 0.025, power = 0.9, alternative = "one.sided")$n
power.t.test(delta = 5,   sd = 18,  sig.level = 0.025, power = 0.9, alternative = "one.sided")$n
ceiling(159 / 0.85)                                   # 탈락 15%를 감안한 인원

### 셀 9. 비율 차이의 비열등성 표본크기와 검정력

In [ ]:
z <- qnorm(0.975) + qnorm(0.80)                       # 단측 α 0.025, 검정력 80%
n <- z^2 * 2 * 0.85 * 0.15 / 0.10^2                   # 두 군 모두 85%, 한계 −10%p
c(n = n, enrolled = ceiling(ceiling(n) / 0.9))
pw <- function(p, n = 201, margin = 0.10) {          # 실제 차이가 0일 때의 검정력
  se <- sqrt(2 * p * (1 - p) / n)
  pnorm((margin - qnorm(0.975) * se) / se)
}
round(sapply(c(0.70, 0.85, 0.95), pw), 3)

## e. 과제 정답

### 셀 10. 과제 1 정답: 한계를 4%p로 좁히면

In [ ]:
ni_t(83.9, 16.8, 304, 85.2, 17.5, 304, margin = 4)        # 한계 4%p
power.t.test(delta = 4, sd = 18, sig.level = 0.025, power = 0.9, alternative = "one.sided")$n

### 셀 11. 과제 2 정답: Cmax의 동등성

In [ ]:
fc <- lm(log(cmax) ~ subject + period + formulation, data = be)
ci <- confint(fc, "formulationT", level = 0.90)
round(100 * exp(c(GMR = unname(coef(fc)["formulationT"]), lower = ci[1], upper = ci[2])), 2)
round(100 * sqrt(exp(sigma(fc)^2) - 1), 1)